# 2. Safety assessment: risk, reliability and inherent safety

**Learning goals**
- Build a risk matrix and rank risk categories (BioSSbD step 2).
- Compute the Heikkilä **Inherent Safety Index (ISI)** and hazardous-inventory KPIs (step 3).
- See how a design change moves the KPIs.

In [ ]:
from pathlib import Path
import pandas as pd
import biossbd
from biossbd import load_system
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
EXAMPLE = Path("..") / "examples" / "acod" / "acod_system.yaml"
system = load_system(EXAMPLE)
print("bioSSbD-py", biossbd.__version__, "|", system.name)

## 2.1 Risk and reliability

In [ ]:
from biossbd.safety import risk_table
risk_table(system.get_alternative("Lactoserum + manure").risks)

In [ ]:
import matplotlib.pyplot as plt
rt = risk_table(system.get_alternative("Lactoserum + manure").risks)
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(rt.likelihood, rt.impact, s=80)
for _, r in rt.iterrows():
    ax.annotate(r.risk, (r.likelihood, r.impact), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set(xlim=(0.5, 5.5), ylim=(0.5, 5.5), xlabel="Likelihood (1-5)", ylabel="Impact (1-5)", title="Risk matrix")
ax.grid(alpha=.3)

## 2.2 Inherent safety KPIs

In [ ]:
from biossbd.safety import safety_kpis
rows = []
for alt in system.alternative_names:
    k = safety_kpis(system, alt)
    rows.append({"alternative": alt, "ISI": k.isi.total, "ISI normalised": k.isi.normalised,
                 "hazardous inventory (kg)": k.hazardous_inventory_kg, "high-risk units": len(k.high_risk_units)})
pd.DataFrame(rows)

In [ ]:
k = safety_kpis(system, "Lactoserum + manure")
pd.Series({**k.isi.chemical, **k.isi.process}, name="score")

## 2.3 A design change: smaller gas holder
Inherent safety principle *minimisation*: halve the biogas holder inventory and compare.

In [ ]:
import copy
s2 = system.model_copy(deep=True)
for u in s2.units:
    if u.name == "Biogas holder":
        u.inventory_kg = 300
before = safety_kpis(system, "Lactoserum + manure")
after = safety_kpis(s2, "Lactoserum + manure")
print("hazardous inventory:", before.hazardous_inventory_kg, "->", after.hazardous_inventory_kg, "kg")
print("ISI:", before.isi.total, "->", after.isi.total)

## Exercise
The ISI did not change in 2.3. Look at the inventory bands in `biossbd.safety.indices.INVENTORY_ISBL_BANDS` and explain why. What inventory would lower the score? Which other KPI captured the improvement?